# Question B4 (10 marks)

> Model degradation is a common issue faced when deploying machine learning models (including neural networks) in the real world. New data points could exhibit a different pattern from older data points due to factors such as changes in government policy or market sentiments. For instance, housing prices in Singapore have been increasing and the Singapore government has introduced 3 rounds of cooling measures over the past years (16 December 2021, 30 September 2022, 27 April 2023).

> In such situations, the distribution of the new data points could differ from the original data distribution which the models were trained on. Recall that machine learning models often work with the assumption that the test distribution should be similar to train distribution. When this assumption is violated, model performance will be adversely impacted.  In the last part of this assignment, we will investigate to what extent model degradation has occurred.


> Your co-investigators used a linear regression model to rapidly test out several combinations of train/test splits and shared with you their findings in a brief report attached in Appendix A below. You wish to investigate whether your deep learning model corroborates with their findings.

In [1]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from alibi_detect.cd import TabularDrift


---

## 1. Model Evaluation

> Evaluate your model from B1 on data from year 2022 and report the test R2.

In [2]:
df = pd.read_csv('hdb_price_prediction.csv')

# Define the columns to be used
cont_cols = [
    "dist_to_nearest_stn"       , 
    "dist_to_dhoby"             ,     
    "degree_centrality"         , 
    "eigenvector_centrality"    , 
    "remaining_lease_years"     , 
    "floor_area_sqm"
]

cat_cols = [
    "month"             , 
    "town"              ,  
    "flat_model_type"   , 
    "storey_range"
]

target_cols = [
    "resale_price"
]

train   = df[df["year"] <= 2019].copy()
val     = df[df["year"] == 2020].copy()
test    = df[df["year"] == 2022].copy()

In [3]:
from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

data_config = DataConfig(
    target                          = target_cols,  # target column name
    continuous_cols                 = cont_cols,   # list of continuous column names to use
    categorical_cols                = cat_cols,    # list of categorical column names to use
    normalize_continuous_features   = True,           # whether to normalize continuous features
) 

trainer_config = TrainerConfig(
    # auto_lr_find    = True,
    batch_size      = 1024,
    max_epochs      = 50,
    seed            = SEED,
)

# Define the Model Architecture
model_config = CategoryEmbeddingModelConfig(
    task        = "regression",
    layers      = "50",     # 3 hidden layer with 5 neurons each
    activation  = "ReLU",
)

optimizer_config = OptimizerConfig(
    optimizer="Adam",
)

tabular_model = TabularModel(
    data_config         = data_config       ,
    model_config        = model_config      ,
    optimizer_config    = optimizer_config  ,
    trainer_config      = trainer_config,
)

2026-03-06 01:27:34,406 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [4]:
import torch

# Capture the original function
_orig_torch_load = torch.load

# Define a version that always sets weights_only to False
def _trusted_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)

# Overwrite the global torch.load
torch.load = _trusted_load

In [5]:
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.preprocessing import StandardScaler

# Define your scaler
target_scaler = StandardScaler()

# Train the model using train/validation split
tabular_model.fit(
    train           =train, 
    validation      =val,
    target_transform=target_scaler
)

# Predict on test set
test_predictions = tabular_model.predict(test)

test_predictions.head()

Seed set to 42
2026-03-06 01:27:34,429 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-06 01:27:34,445 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-06 01:27:34,498 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-06 01:27:34,624 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-06 01:27:34,678 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/leyanzhi/Rep

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  2.9 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.5 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.5 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_conne
ctor.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: 
UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_conne
ctor.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

2026-03-06 01:27:57,268 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-06 01:27:57,268 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


,resale_price_prediction
116427,211538.90625
116428,309828.90625
116429,266989.65625
116430,296500.56250
116431,338435.06250


In [6]:
# Get prediction column (usually "resale_price_prediction")
predicted_cols = [col for col in test_predictions.columns if col.endswith("_prediction")]
predicted_col = predicted_cols[0]  # one target column

y_true = test[target_cols[0]].to_numpy()
y_pred = test_predictions[predicted_col].to_numpy()

# Metrics
test_mse = mean_squared_error(y_true, y_pred)
test_r2 = r2_score(y_true, y_pred)

print(f"Test MSE: {test_mse:.4f}")
print(f"Test R2 : {test_r2:.4f}")

Test MSE: 16063739199.7735
Test R2 : 0.4459


> Evaluate your model from B1 on data from year 2023 and report the test R2.

In [7]:
test2 = df[df["year"] == 2023].copy()

# Predict on test set
test_predictions2 = tabular_model.predict(test2)

test_predictions2.head()

,resale_price_prediction
143129,207449.390625
143130,221032.921875
143131,206322.890625
143132,209685.531250
143133,211673.718750


In [8]:
predicted_cols2 = [col for col in test_predictions2.columns if col.endswith("_prediction")]
predicted_col2 = predicted_cols2[0]  # one target column

y_true2 = test2[target_cols[0]].to_numpy()
y_pred2 = test_predictions2[predicted_col2].to_numpy()

test2_mse = mean_squared_error(y_true2, y_pred2)
test2_r2 = r2_score(y_true2, y_pred2)

print(f"Test2 MSE: {test2_mse:.4f}")
print(f"Test2 R2 : {test2_r2:.4f}")

Test2 MSE: 24807205235.2917
Test2 R2 : 0.1585


> Did model degradation occur for the deep learning model?


I have set `auto_lr_find` to false and using the default lr as otherwise the result for R2 is really bad.

Yes, R2 dropped from 0.4459 to 0.1585



---




## 2. Drifted Features

> Model degradation could be caused by [various data distribution shifts](https://huyenchip.com/2022/02/07/data-distribution-shifts-and-monitoring.html#data-shift-types): covariate shift (features), label shift and/or concept drift (altered relationship between features and labels).
There are various conflicting terminologies in the [literature](https://www.sciencedirect.com/science/article/pii/S0950705122002854#tbl1). Let’s stick to this reference for this assignment.

> Using the **Alibi Detect** library, apply the **TabularDrift** function with the training data (year 2019 and before) used as the reference and **detect which features have drifted** in the 2023 test dataset. Before running the statistical tests, ensure you **sample 800 data points** each from the train and test data. Do not use the whole train/test data. (Hint: use this example as a guide https://docs.seldon.io/projects/alibi-detect/en/stable/examples/cd_chi2ks_adult.html)


The `TabularDrift function` in Alibi Detect is a "two-sample" test wrapper. It applies different statistical tests based on the data type:

- Kolmogorov-Smirnov (KS) Test: Used for continuous features (`floor_area_sqm`, etc.). It measures the maximum distance between the cumulative distributions of the two samples.

- Chi-Squared Test: Used for categorical features (`town`, `month`, etc.). It checks if the frequencies of categories have changed significantly.

Covariate Shift: If `is_drift` is 1 (True), it means the distribution of that specific feature has changed significantly between 2019 and 2023. This is a classic Covariate Shift.

The Baseline Effect: Because your reference is 2019 and your test is 2023, features related to time (like `remaining lease`) will almost certainly drift because 4 years have passed for every single flat in the dataset.

P-Value Interpretation: A very low p-value (e.g., $p < 0.001$) suggests that the difference between 2019 and 2023 is not due to random sampling luck, but a genuine shift in the Singapore housing market landscape.

In [9]:
from sklearn.preprocessing import LabelEncoder

feature_cols = cont_cols + cat_cols # List of all features to be used for drift detection

# Filter for the specific years
train_df        = df[df["year"] <= 2019][feature_cols].copy()   # reference data from 2019 and earlier
test_2023_df    = df[df["year"] == 2023][feature_cols].copy()   # test data from 2023

# Sampling
train_sample    = train_df.sample(n=800, random_state=SEED).copy()
test_sample     = test_2023_df.sample(n=800, random_state=SEED).copy()

# TabularDrift expects a dictionary where the keys are the column indices of the categorical features 
#   and the values are the number of categories in those features.
categories_per_feature = {}

for col in cat_cols:

    label_encoder = LabelEncoder()

    # Fit ONLY on the categories that actually exist in these 1600 rows
    # If we fit on the entire dataset, we might include categories that are not present in our samples
    # Alibi Detect build a contigency table with slots for every single category
    # In chi-squared test for categorical features, if a category is absent in the sample, 
    #   the slot for that category will be 0, leading to division by zeros in frequency calculation
    combined_data = pd.concat([train_sample[col], test_sample[col]])
    label_encoder.fit(combined_data)
    
    train_sample[col]   = label_encoder.transform(train_sample[col])
    test_sample[col]    = label_encoder.transform(test_sample[col])
    
    # Store the exact number of categories present in our samples for Alibi Detect
    col_idx                         = feature_cols.index(col)
    num_categories                  = len(label_encoder.classes_)
    categories_per_feature[col_idx] = num_categories


X_ref = train_sample.values
X_test_2023 = test_sample.values

print("Categories per feature (column index: num categories):")
for idx, num_cat in categories_per_feature.items():
    print(f"Column Index {idx} ({feature_cols[idx]}): {num_cat} categories")

print ("Reference data shape:", X_ref.shape)
print ("Test data shape:", X_test_2023.shape)

Categories per feature (column index: num categories):
Column Index 6 (month): 12 categories
Column Index 7 (town): 26 categories
Column Index 8 (flat_model_type): 33 categories
Column Index 9 (storey_range): 14 categories
Reference data shape: (800, 10)
Test data shape: (800, 10)


In [10]:
# Initialize TabularDrift
cd = TabularDrift(
    X_ref, 
    p_val=0.05, 
    categories_per_feature=categories_per_feature
)

# Run Prediction
drift_preds = cd.predict(X_test_2023)

# Identify which features drifted
features_drifted    = []
p_vals              = drift_preds['data']['p_val']
threshold           = drift_preds['data']['threshold']

for i, feature in enumerate(feature_cols):
    p_value = p_vals[i]
    is_drifted = bool(p_value < threshold)
    if is_drifted:
        features_drifted.append(feature)
    print(f"Feature: {feature:25} | Drifted: {str(is_drifted):5} | p-value: {p_value:.4f}")

print(f"\nTotal features with detected drift: {len(features_drifted)}")
print(f"Overall drift detected: {bool(drift_preds['data']['is_drift'])}")

Feature: dist_to_nearest_stn       | Drifted: False | p-value: 0.4163
Feature: dist_to_dhoby             | Drifted: False | p-value: 0.0155
Feature: degree_centrality         | Drifted: False | p-value: 0.6556
Feature: eigenvector_centrality    | Drifted: False | p-value: 0.0284
Feature: remaining_lease_years     | Drifted: True  | p-value: 0.0000
Feature: floor_area_sqm            | Drifted: False | p-value: 0.1078
Feature: month                     | Drifted: True  | p-value: 0.0000
Feature: town                      | Drifted: False | p-value: 0.0257
Feature: flat_model_type           | Drifted: False | p-value: 0.0116
Feature: storey_range              | Drifted: False | p-value: 0.0758

Total features with detected drift: 2
Overall drift detected: True


`remaining_lease_years` and `month` are the 2 features that have shifted

> 5. Assuming that the flurry of housing measures have made an impact on the relationship between all the features and resale_price (i.e. $P(Y|X)$ changes), which type of data distribution shift possibly led to model degradation?


We have 3 type of drifts to choose from:

1. **Covariate Drift**

    Covariate is an independent variable that is related to the outcome but is not the main focus of the research. For example, if we want to find how location affect house price, location is the explantory variable, house price is response variable. sqm is our possible covariate, as although sqm is not the focus of the research, we know that it affects the house price too.

    The distribution of your input features $P(X)$ changes, but the relationship between the features and the target $P(Y|X)$ stays the same

    For example, for `remaining_lease_years`, the lease years in the test data are shorter than that of train data (since 3 years have passed), but buyers still see the older flats with same value. (i.e. if older flats became more valuable or less valuable after covid, then this is NOT covariate drift)

    This is exactly what the Alibi Detect `TabularDrift` function tests for.

2. **Label Drift**

    Also known as prior shift, prior probability shift or target shift. Prior is initial belief before anything new is observed i.e. $P(y)$, while posterior is the updated belief after something is observed i.e. $P(y|X)$.

    The distribution of your target variable $P(Y)$ changes, but the conditional distribution of features given the target $P(X|Y)$ stays the same. 

    For example, if due to inflation or such, all HDBs see an increase in resale price. But the type of HDBs sold given a price did not change (e.g. if 1M in 2019 get you a particular flat and 1M in 2023 get you the same flat, even though flat price has generally increased, for eg we have included more of the expensive flats in our test dataset)

3. **Concept Drift**

    Also known as posterior shift

    The fundamental relationship between the input features and the target variable $P(Y|X)$ changes. The features might look exactly the same, but the model's old rules for mapping those features to a price are now wrong.


From the definitions, it is obviously **Concept Drift**, since question explicity says $P(Y|X)$ changed.

> 6. From your analysis via TabularDrift, which features contribute to this shift?


`TabularDrift` only detects **Covariate Shift**, not Concept Drift (it only accepeted feature columns and completely ignored label columns)

`remaining_lease_years` and `month` are the 2 features that contributed to Covariate Shift

> 7. Suggest 1 way to address model degradation and implement it, showing improved test R2 for year 2023.


The most robust and industry-standard way to address model degradation caused by Concept Drift and Covariate Shift is Periodic Retraining (also known as Continuous Training).

We shift our training window to 2021, validate on 2022, and test on 2023.

In [11]:
train3   = df[df["year"] == 2021].copy()
val3     = df[df["year"] == 2022].copy()
test3    = df[df["year"] == 2023].copy()

print ("Train 2021 shape:", train3.shape)
print ("Val 2022 shape:", val3.shape)
print ("Test 2023 shape:", test3.shape)

Train 2021 shape: (29057, 14)
Val 2022 shape: (26702, 14)
Test 2023 shape: (16424, 14)


In [12]:
tabular_model3 = TabularModel(
    data_config         = data_config       ,
    model_config        = model_config      ,
    optimizer_config    = optimizer_config  ,
    trainer_config      = trainer_config,
)

# Train the model using train/validation split
tabular_model3.fit(
    train           =train3, 
    validation      =val3,
    target_transform=target_scaler
)

# Predict on test set
test_predictions3 = tabular_model3.predict(test3)

test_predictions3.head()

2026-03-06 01:27:57,506 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-03-06 01:27:57,510 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-06 01:27:57,516 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-06 01:27:57,539 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-06 01:27:57,548 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-06 01:27:57,553 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/si

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  3.0 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.6 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.6 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.6 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: 
UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_conne
ctor.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 
value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

/opt/miniconda3/envs/sc4001-assignment/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:317: The 
number of training batches (29) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower 
value for log_every_n_steps if you want to see logs for the training epoch.

2026-03-06 01:28:07,930 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-06 01:28:07,930 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


,resale_price_prediction
143129,106033.62500
143130,212196.71875
143131,192292.53125
143132,196095.96875
143133,98135.93750


In [13]:
# Get prediction column (usually "resale_price_prediction")
predicted_cols = [col for col in test_predictions3.columns if col.endswith("_prediction")]
predicted_col = predicted_cols[0]  # one target column

y_true = test3[target_cols[0]].to_numpy()
y_pred = test_predictions3[predicted_col].to_numpy()

# Metrics
test_mse = mean_squared_error(y_true, y_pred)
test_r2 = r2_score(y_true, y_pred)

print(f"Test MSE: {test_mse:.4f}")
print(f"Test R2 : {test_r2:.4f}")

Test MSE: 12866144491.4030
Test R2 : 0.5636


R2 improved from 0.1585 to 0.5636.

---

### Appendix A



Here are our results from a linear regression model. We used StandardScaler for continuous variables and OneHotEncoder for categorical variables.

While 2021 data can be predicted well, test R2 dropped rapidly for 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| Year <= 2020 | 2021     | 0.76    |
| Year <= 2020 | **2022**     | 0.41    |
| Year <= 2020 | **2023**     | **0.10**   |



Similarly, a model trained on 2017 data can predict 2018-2021 well (with slight degradation in performance for 2021), but drops drastically in 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2017         | 2018     | 0.90    |
|              | 2019     | 0.89    |
|              | 2020     | 0.87    |
|              | 2021     | 0.72    |
|              | **2022**     | **0.37**    |
|              | **2023**     | **0.09**    |

With the test set fixed at year 2021, training on data from 2017-2020 still works well on the test data, with minimal degradation. Training sets closer to year 2021 generally do better.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2020         | 2021     | 0.81    |
| 2019         | 2021     | 0.75    |
| 2018         | 2021     | 0.73    |
| 2017         | 2021     | 0.72    |